# E28 — o teto do que se aprende

O capítulo da tolerância conta quantas explicações cabem: duas de cento e oitenta. Esta medição
pergunta de que a contagem depende, variando as duas coisas que aquele capítulo fixou — **quantas
peças foram tentadas** e **quantos dias foram vistos**.

In [1]:
# <- brinque com: COMPRIMENTOS, GRADES, SEMENTE
import json
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import frevolab
from frevolab import evidencia, graficos

RAIZ = Path.cwd()
COMPRIMENTOS = (500, 1000, 2000, 4000, 6718)
GRADES = ((0.05, 0.25), (0.02, 0.08, 0.18), (0.02, 0.05, 0.08, 0.12, 0.18, 0.25))
SEMENTE = evidencia.SEMENTE_PADRAO

print("frevolab %s | comprimentos %s | grades de %s pecas"
      % (frevolab.VERSAO, COMPRIMENTOS, [len(g) * len(evidencia.GRADE_RAZAO_PADRAO) for g in GRADES]))

frevolab 0.1.0 | comprimentos (500, 1000, 2000, 4000, 6718) | grades de [12, 18, 36] pecas


In [2]:
# A contagem contra as duas coisas: quantos dias, e quantas pecas tentadas.
linhas = []
for grade in GRADES:
    for dias in COMPRIMENTOS:
        r = evidencia.contagem(dias, grade_p=grade, semente=SEMENTE)
        linhas.append({"grade_p": len(grade), "tentadas": r["tentadas"], "dias": r["dias"],
                       "cabem": r["cabem"], "fracao_pct": r["fracao_pct"]})
quadro = pd.DataFrame(linhas)
print(quadro.to_string(index=False))
print()
print("mesma grade, mais dias: %s" % [(int(l["dias"]), l["cabem"]) for l in linhas if l["tentadas"] == 180])
print("mesmos dias, mais pecas: %s" % [(l["tentadas"], l["cabem"]) for l in linhas if l["dias"] == 6718])

 grade_p  tentadas  dias  cabem  fracao_pct
       2        60   500      7   11.666667
       2        60  1000      0    0.000000
       2        60  2000      5    8.333333
       2        60  4000      1    1.666667
       2        60  6718      1    1.666667
       3        90   500     12   13.333333
       3        90  1000      2    2.222222
       3        90  2000      5    5.555556
       3        90  4000      0    0.000000
       3        90  6718      3    3.333333
       6       180   500     19   10.555556
       6       180  1000      2    1.111111
       6       180  2000     10    5.555556
       6       180  4000      8    4.444444
       6       180  6718      2    1.111111

mesma grade, mais dias: [(500, 19), (1000, 2), (2000, 10), (4000, 8), (6718, 2)]
mesmos dias, mais pecas: [(60, 1), (90, 3), (180, 2)]


In [3]:
# Figura 1: a contagem contra os dias, uma curva por grade.
fig, eixo = plt.subplots(figsize=(8.4, 4.2))
for grade, cor in zip(GRADES, ("#b03a2e", "#2e7d32", "#1f4e79")):
    linha = [l for l in linhas if l["grade_p"] == len(grade)]
    eixo.plot([l["dias"] for l in linha], [l["cabem"] for l in linha], marker="o", lw=1.7, color=cor,
              label="%d peças tentadas" % linha[0]["tentadas"])
eixo.set_xscale("log")
eixo.set_xlabel("dias vistos")
eixo.set_ylabel("explicações que cabem na tolerância")
eixo.legend(frameon=False, fontsize=9)
eixo.grid(alpha=0.25, which="both", ls=":")
fig.tight_layout()
graficos.salvar(fig, "E28_teto", 1)
plt.close(fig)
print("figura gravada")

figura gravada


## Leitura visual das figuras

Feita nesta sessão abrindo o .png com a ponte de visão (AGENTS.md §9), depois de o caderno rodar.

**Pendente**: a figura existe e o caderno rodou, mas o desenho ainda não foi aberto. Nada nesta
célula é afirmação sobre ele até essa passagem ser feita.

In [4]:
# O resultado: um objeto por grandeza, para o livro citar por comando.
NOMES_GRADE = {2: "dois", 3: "tres", 6: "seis"}
NOMES_DIAS = {500: "quinhentos", 1000: "mil", 2000: "dois_mil", 4000: "quatro_mil",
              6718: "seis_mil_setecentos_e_dezoito"}
resultado = {
    "teto_comprimentos": int(len(COMPRIMENTOS)),
    "teto_grades": int(len(GRADES)),
    "teto_semente": int(SEMENTE),
    "teto_dias_menor": int(min(COMPRIMENTOS)),
    "teto_dias_maior": int(max(COMPRIMENTOS)),
    "teto_dias_meio": int(sorted(COMPRIMENTOS)[2]),
}
for grade in GRADES:
    nome = NOMES_GRADE[len(grade)]
    for dias in COMPRIMENTOS:
        r = next(l for l in linhas if l["grade_p"] == len(grade) and l["dias"] == dias)
        resultado["teto_%s_%s" % (nome, NOMES_DIAS[dias])] = int(r["cabem"])
resultado["teto_pecas_menor"] = int(min(l["tentadas"] for l in linhas))
resultado["teto_pecas_maior"] = int(max(l["tentadas"] for l in linhas))
caminho = Path("lab/resultados/E28_teto.json")
caminho.write_text(json.dumps(resultado, indent=1, ensure_ascii=False, sort_keys=True), encoding="utf-8")
print("%s gravado | %d grandezas" % (caminho, len(resultado)))

lab/resultados/E28_teto.json gravado | 23 grandezas
